In [ ]:
import os, sys

%cd /content
repo_url = "https://github.com" + "/comfyanonymous/ComfyUI"
!git clone {repo_url} /content/ComfyUI
%cd /content/ComfyUI
sys.path.insert(0, '/content/ComfyUI')

!pip install -q torchsde einops diffusers accelerate xformers
!apt -y install -qq aria2

!mkdir -p models/unet models/vae models/clip
hf_base = "https://huggingface.co"
u_unet = hf_base + "/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors"
u_vae = hf_base + "/camenduru/FLUX.1-dev/resolve/main/ae.sft"
u_clip = hf_base + "/camenduru/FLUX.1-dev/resolve/main/clip_l.safetensors"
u_t5 = hf_base + "/camenduru/FLUX.1-dev/resolve/main/t5xxl_fp8_e4m3fn.safetensors"

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {u_unet} -d /content/ComfyUI/models/unet -o flux1-schnell.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {u_vae} -d /content/ComfyUI/models/vae -o ae.sft
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {u_clip} -d /content/ComfyUI/models/clip -o clip_l.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {u_t5} -d /content/ComfyUI/models/clip -o t5xxl_fp8_e4m3fn.safetensors

import random
import torch
import numpy as np
from PIL import Image
import nodes
from nodes import NODE_CLASS_MAPPINGS
from comfy_extras import nodes_custom_sampler
from comfy import model_management

DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
RandomNoise = nodes_custom_sampler.NODE_CLASS_MAPPINGS["RandomNoise"]()
BasicGuider = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicGuider"]()
KSamplerSelect = nodes_custom_sampler.NODE_CLASS_MAPPINGS["KSamplerSelect"]()
BasicScheduler = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicScheduler"]()
SamplerCustomAdvanced = nodes_custom_sampler.NODE_CLASS_MAPPINGS["SamplerCustomAdvanced"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("t5xxl_fp8_e4m3fn.safetensors", "clip_l.safetensors", "flux")
    unet = UNETLoader.load_unet("flux1-schnell.safetensors", "fp8_e4m3fn")
    vae = VAELoader.load_vae("ae.sft")

def closestNumber(n, m):
    q = int(n / m)
    n1 = m * q
    n2 = m * (q + 1) if (n * m) > 0 else m * (q - 1)
    return n1 if abs(n - n1) < abs(n - n2) else n2

In [ ]:
with torch.inference_mode():
    positive_prompt = "black forest toast spelling out the words 'FLUX SCHNELL'"
    width = 1024
    height = 1024
    seed = 0
    steps = 4
    sampler_name = "euler"
    scheduler = "simple"

    if seed == 0:
        seed = random.randint(0, 18446744073709551615)
    print(f"Using Seed: {seed}")

    cond, pooled = clip.encode_from_tokens(clip.tokenize(positive_prompt), return_pooled=True)
    cond = [[cond, {"pooled_output": pooled}]]
    noise = RandomNoise.get_noise(seed)
    guider = BasicGuider.get_guider(unet, cond)
    sampler = KSamplerSelect.get_sampler(sampler_name)
    sigmas = BasicScheduler.get_sigmas(unet, scheduler, steps, 1.0)
    latent_image = EmptyLatentImage.generate(closestNumber(width, 16), closestNumber(height, 16))
    sample, sample_denoised = SamplerCustomAdvanced.sample(noise, guider, sampler, sigmas, latent_image)
    model_management.soft_empty_cache()
    decoded = VAEDecode.decode(vae, sample).detach()
    Image.fromarray(np.array(decoded*255, dtype=np.uint8)).save("/content/flux.png")

Image.fromarray(np.array(decoded*255, dtype=np.uint8))